In [ ]:
library("sva") #Note this exercise requires sva (>= v3.36.0) which is only available for R (>= v4.x)
library("ggplot2")
library("gridExtra")
library("edgeR")
library("UpSetR")
library(grid)
library("readr")
library("DESeq2")
library("tximport")
library("readr")
library("arrow")
library("SummarizedExperiment")
library("UpSetR")
library("ggplot2")
library(ComplexUpset)
library('edgeR')
library(ggplot2)
library(EnhancedVolcano)
library(dplyr)
#library(svglite)

In [ ]:
# get version of edgeR
packageVersion("edgeR")

In [ ]:
# get the version of R
R.version.string

In [ ]:
uncorrected_data = read.table("nf-core-plantlongrnaseq/output_desiree_unitato/tximeta/all_samples.gene_counts.tsv", header = TRUE, sep = "\t", as.is = c(1,2))
sample_names = names(uncorrected_data)[3:length(names(uncorrected_data))]


In [ ]:
#review data structure
head(uncorrected_data)
dim(uncorrected_data)

In [ ]:
library_methods = c("1", "1", "1", "1", "2", "2", "1", "2")    
conditions = rep(c("control", "heat"), c(4,4))
replicates = sample_names

#calculate principal components for the uncorrected data
pca_uncorrected_obj = prcomp(uncorrected_data[,sample_names])

#pull PCA values out of the PCA object
pca_uncorrected = as.data.frame(pca_uncorrected_obj[2]$rotation)

#determine the percentage of variance explained by each PC
variance = (pca_uncorrected_obj$sdev)^2
percent_variance = (variance / sum(variance)) * 100

#assign labels to the data frame
pca_uncorrected[,"condition"] = conditions
pca_uncorrected[,"library_method"] = library_methods
pca_uncorrected[,"replicate"] =replicates

#plot the PCA
#create a classic 2-dimension PCA plot (first two principal components) with conditions and library methods indicated
pc1_label = paste("PC1 (", round(percent_variance[1], digits=1), "% variance explained)", sep="")
pc2_label = paste("PC2 (", round(percent_variance[2], digits=1), "% variance explained)", sep="")

cols <- c("UHR" = "#481567FF", "HBR" = "#1F968BFF")
p1 = ggplot(data = pca_uncorrected, aes(x = PC1, y = PC2, color = condition, shape = library_method))
p1 = p1 + geom_point(size = 3)
#p1 = p1 + stat_ellipse(type = "norm", linetype = 2)
p1 = p1 + labs(title = "PCA, RNA-seq counts for heat vs control Kinnex (uncorrected data)", color = "Condition", shape="Library Method")
p1 = p1 + labs(x = pc1_label, y = pc2_label)
#p1 = p1 + scale_colour_manual(values = cols)

# preform batch correction using ComBat-Seq

In [ ]:
#perform the batch correction

groups = sapply(as.character(conditions), switch, "control" = 1, "heat" = 2, USE.NAMES = FALSE)
batches = sapply(as.character(library_methods), switch, "1" = 1, "2" = 2, USE.NAMES = FALSE)

#now run ComBat_seq
corrected_data = ComBat_seq(counts = as.matrix(uncorrected_data[,sample_names]), batch = batches, group = groups)

#join the gene and chromosome names onto the now corrected counts from ComBat_seq
corrected_data = as.data.frame(cbind(uncorrected_data[, c("gene_id", "gene_name")], corrected_data))

#compare dimensions of corrected and uncorrected data sets
dim(uncorrected_data)
dim(corrected_data)

#visually compare values of corrected and uncorrected data sets
head(uncorrected_data)
head(corrected_data)
#rename the first column of the corrected data to "gene_id" to match uncorrected data
colnames(corrected_data)[1] = "gene_id"
colnames(corrected_data)[2] = "gene_name"
# make the counts to dbl
corrected_data[, sample_names] = lapply(corrected_data[, sample_names], as.numeric)


In [ ]:
write.table(corrected_data, file = "nf-core-plantlongrnaseq/output_desiree_unitato/tximeta/all_samples.gene_counts_combat_seq_corrected.tsv", sep = "\t", row.names = FALSE, quote = FALSE)

#### check batch gene

In [ ]:
corrected_data[uncorrected_data$gene_id == "Soltu.DM.01G000030",]
# plot corrected counts for a specific gene


In [ ]:
uncorrected_data[uncorrected_data$gene_id == "Soltu.DM.01G000030",  ]

### check heat gene

In [ ]:
corrected_data[uncorrected_data$gene_id == "Soltu.DM.08G012670",]

In [ ]:
uncorrected_data[uncorrected_data$gene_id == "Soltu.DM.08G012670",  ]

In [ ]:
#calculate principal components for the uncorrected data
pca_corrected_obj = prcomp(corrected_data[, sample_names])

#pull PCA values out of the PCA object
pca_corrected = as.data.frame(pca_corrected_obj[2]$rotation)

#determine the percentage of variance explained by each PC
variance = (pca_corrected_obj$sdev)^2
percent_variance = (variance / sum(variance)) * 100

#assign labels to the data frame
pca_corrected[,"condition"] = conditions
pca_corrected[,"library_method"] = library_methods
pca_corrected[,"replicate"] = replicates

#as above, create a PCA plot for comparison to the uncorrected data
pc1_label = paste("PC1 (", round(percent_variance[1], digits=1), "% variance explained)", sep="")
pc2_label = paste("PC2 (", round(percent_variance[2], digits=1), "% variance explained)", sep="")
cols <- c("UHR" = "#481567FF", "HBR" = "#1F968BFF")

p2 = ggplot(data = pca_corrected, aes(x = PC1, y = PC2, color = condition, shape = library_method))
p2 = p2 + geom_point(size = 3)
#p2 = p2 + stat_ellipse(type = "norm", linetype = 2)
p2 = p2 + labs(title = "(batch corrected data)", color = "Condition", shape = "Library Method")
p2 = p2 + labs(x = pc1_label, y = pc2_label)
#p2 = p2 + scale_colour_manual(values = cols)

#pdf(file = "Uncorrected-vs-BatchCorrected-PCA.pdf")
grid.arrange(p1, p2, nrow = 2)
dev.off()

In [ ]:
# Set up color scheme

keep <- rowSums(cpm(corrected_data[, sample_names]) > 4) >= 4
filtered_corrected <- corrected_data[keep, ]
keep <- rowSums(cpm(uncorrected_data[, sample_names]) > 4) >= 4
filtered_uncorrected <- uncorrected_data[keep, ]
group <- conditions
custom_palette <- rainbow(length(unique(group)))
col <- custom_palette[as.numeric(as.factor(group))]
# Create a 2x2 layout
par(mfrow = c(2, 2), bg = "white")

# Plot 1: Corrected data before filtering
boxplot(cpm(corrected_data[, sample_names], log = TRUE), 
        las = 2, 
        ylab = "log(scaled counts)", 
        main = "Corrected - Before filtering", 
        col = col)


boxplot(cpm(filtered_corrected[, sample_names], log = TRUE), 
        las = 2, 
        ylab = "log(scaled counts)", 
        main = "Corrected - After filtering", 
        col = col)

# Plot 3: Uncorrected data before filtering
boxplot(cpm(uncorrected_data[, sample_names], log = TRUE), 
        las = 2, 
        ylab = "log(scaled counts)", 
        main = "Uncorrected - Before filtering", 
        col = col)

# Plot 4: Uncorrected data after filtering

boxplot(cpm(filtered_uncorrected[, sample_names], log = TRUE), 
        las = 2, 
        ylab = "log(scaled counts)", 
        main = "Uncorrected - After filtering", 
        col = col)

# Reset layout
par(mfrow = c(1, 1))

In [ ]:
# Function to create density plot
plot_density <- function(data, sample_names, title, col, group) {
  cpm_data <- cpm(data[, sample_names], log = TRUE)
  plot(density(cpm_data[, 1]), lwd = 2,
       main = title, 
       xlab = "Log-cpm", 
       col = col[1],
       ylim = range(sapply(1:ncol(cpm_data), function(i) range(density(cpm_data[, i])$y))))
  for (i in 2:ncol(cpm_data)) {
    lines(density(cpm_data[, i]), col = col[i], lwd = 2)
  }
  legend("topright", legend = unique(group), 
         col = rainbow(length(unique(group))), lwd = 2, cex = 0.7)
}

# Set up color scheme
group <- conditions
custom_palette <- rainbow(length(unique(group)))
col <- custom_palette[as.numeric(as.factor(group))]

# Create 2x2 layout
par(mfrow = c(2, 2), bg = "white")

# Plot all four
plot_density(corrected_data, sample_names, "Corrected - Unfiltered", col, group)



plot_density(filtered_corrected, sample_names, "Corrected - Filtered", col, group)

plot_density(uncorrected_data, sample_names, "Uncorrected - Unfiltered", col, group)


plot_density(filtered_uncorrected, sample_names, "Uncorrected - Filtered", col, group)

# Reset layout
par(mfrow = c(1, 1))

In [ ]:
dev.off()
par(bg = "white", mfrow = c(2, 2))
plotMDS(corrected_data[, sample_names], labels = conditions, col = col, cex = 0.6, 
        main = "MDS plot batch corrected counts")
plotMDS(uncorrected_data[, sample_names], labels = conditions, col = col, cex = 0.6, 
        main = "MDS plot uncorrected counts")

# perform differential expression analysis on the uncorrected data and batch corrected data sets

## Limma with batch corrected

In [ ]:
#colnames(corrected_data) = corrected_data$gene_id
count_matrix <- corrected_data[, sample_names]
rownames(count_matrix) <- corrected_data$gene_id
dge_lr_corr <- DGEList(counts=count_matrix, group = conditions, samples = replicates, genes = corrected_data["gene_id"])
dge_filtered_lr_corr <- dge_lr_corr[rowSums(cpm(dge_lr_corr) > 3) >= 5, , keep.lib.sizes=FALSE]

logFCcut <- 1 # maybe higher?
adjPcut <- 0.01

par(bg = "white")
design <- model.matrix(~0 + group)
colnames(design) <- levels(as.factor(group))
contrasts <- makeContrasts(
  # Treatment vs Control at different timepoints
  heat - control,
  
  # Add more contrasts as needed for your analysis
  levels = design
)
dge_filtered_lr_corr$design <- design
fit <- edgeR::voomLmFit(dge_filtered_lr_corr, design, plot=TRUE)
fit <- contrasts.fit(fit, contrasts)
fit <- eBayes(fit)

# Determine significant genes
dt_voomlr_corr <- decideTests(fit, lfc = logFCcut, p.value = adjPcut)
res_voomLmFit_lr_corr <- topTable(fit, sort.by = "P", n = Inf)

# Plot 2: voom
barplot(summary(dt_voomlr_corr)[c(1, 3), ], beside = TRUE, las = 2, 
        legend = TRUE, col = c("darkgreen", "darkred"),
        main = "voom")

library(EnhancedVolcano)
EnhancedVolcano(res_voomLmFit_lr_corr,
    lab = rownames(res_voomLmFit_lr_corr),
    x = 'logFC',
    y = 'P.Value',
    pCutoff = 0.01,
    FCcutoff = 1,
    title = "voom lm fit")


# export the results to a CSV file
write.csv(res_voomLmFit_lr_corr, file = "LR_DESIREE_PAPER/ANALYSIS/final_tables_paper/lr_bathc_corr_voom_lm_fit_results.csv", row.names = TRUE)

## Limma with uncorrected

In [ ]:

#colnames(corrected_data) = corrected_data$gene_id
sample_names = c("DE_C_H_13", "DE_C_H_14", "DE_C_H_15", "DE_C_H_16",
                 "DE_H_H_17", "DE_H_H_18",  "DE_H_H_19", "DE_H_H_20")
count_matrix <- uncorrected_data[, sample_names]

replicates = sample_names
conditions = rep(c("control", "heat"), c(4,4))
group = rep(c("control", "heat"), c(4,4))
rownames(count_matrix) <- uncorrected_data$gene_id
dge_lr_uncorr  <- DGEList(counts=count_matrix, group = conditions, samples = replicates, genes = uncorrected_data["gene_id"])
dge_filtered_lr_uncorr  <- dge_lr_uncorr[rowSums(cpm(dge_lr_uncorr) > 3) >= 5,  keep.lib.sizes=FALSE]

logFCcut <- 1 # maybe higher?
adjPcut <- 0.01

par(bg = "white")
design <- model.matrix(~0 + group)
colnames(design) <- levels(as.factor(group))
contrasts <- makeContrasts(
  # Treatment vs Control at different timepoints
  heat - control,
  
  # Add more contrasts as needed for your analysis
  levels = design
)
dge_filtered_lr_uncorr$design <- design
fit <- edgeR::voomLmFit(dge_filtered_lr_uncorr, design, plot=TRUE)
fit <- contrasts.fit(fit, contrasts)
fit <- eBayes(fit)

# Determine significant genes
dt_voomlr_uncorr  <- decideTests(fit, lfc = logFCcut, p.value = adjPcut)
res_voomLmFit_lr_uncorr <- topTable(fit, sort.by = "P", n = Inf)

# Plot 2: voom
barplot(summary(dt_voomlr_uncorr)[c(1, 3), ], beside = TRUE, las = 2, 
        legend = TRUE, col = c("darkgreen", "darkred"),
        main = "voom")

library(EnhancedVolcano)
EnhancedVolcano(res_voomLmFit_lr_uncorr,
    lab = rownames(res_voomLmFit_lr_uncorr),
    x = 'logFC',
    y = 'P.Value',
    pCutoff = 0.01,
    FCcutoff = 1,
    title = "voom lm fit")

## Lima short read -rna seq



In [ ]:
setup_file_paths <- function(base_dir, quant_type, sample_names) {

  
  if (quant_type == "salmon" ) {
    # Salmon output structure: salmon/SAMPLE_NAME/quant.sf
    quant_files <- file.path(base_dir, "salmon", sample_names, "quant.sf")
    
    
  } else if (quant_type == "kallisto") {
    # Kallisto output structure: kallisto/SAMPLE_NAME/abundance.tsv
    quant_files <- file.path(base_dir, "kallisto", sample_names, "abundance.tsv")
    
    
  } else if (quant_type == "oarfish") {
    # Oarfish output structure: oarfish/SAMPLE_NAME/quant.sf
    quant_files <- file.path(base_dir, "oarfish", sample_names, "quant.sf")

  } else {
    stop(paste("Unsupported quantification type:", quant_type))
  }
  
  return(quant_files)
}

In [ ]:

#colnames(corrected_data) = corrected_data$gene_id
samples = data.frame(
  run = c("DE_C_H_13", "DE_C_H_14", "DE_C_H_15", "DE_C_H_16",
          "DE_H_H_17", "DE_H_H_18", "DE_H_H_19", "DE_H_H_20"),
  stringsAsFactors = FALSE
)


base_dir = "Desiree_ASE/output/nf_core_rnaseq_short_read_heat_unitato"
quant_type = "salmon"
sample_df = samples
conditions = rep(c("control", "heat"), each = 4)
tx2gene_file = "Desiree_ASE/output/nf_core_rnaseq_short_read_heat_unitato/salmon/tx2gene.tsv"

group <- factor(conditions)
design <- model.matrix(~0 + group)
colnames(design) <- levels(as.factor(group))
files<- setup_file_paths(base_dir, quant_type, sample_df$run)

tx2gene <- read_tsv(tx2gene_file)
    
txi <- tximport(files, type = quant_type, tx2gene = tx2gene)
colnames(txi$counts) <- samples$run
dge_sr <- DGEList(counts=txi$counts, group = conditions, samples = sample_df$run)

dge_filtered_sr <- dge_sr[rowSums(cpm(dge_sr) > 3) >= 5, , keep.lib.sizes=FALSE]

logFCcut <- 1 # maybe higher?
adjPcut <- 0.01

par(bg = "white")
design <- model.matrix(~0 + group)
colnames(design) <- levels(as.factor(group))
contrasts <- makeContrasts(
  # Treatment vs Control at different timepoints
  heat - control,
  
  # Add more contrasts as needed for your analysis
  levels = design
)
dge_filtered_sr$design <- design
fit <- edgeR::voomLmFit(dge_filtered_sr, design, plot=TRUE)
fit <- contrasts.fit(fit, contrasts)
fit <- eBayes(fit)

# Determine significant genes
dt_voom_sr <- decideTests(fit, lfc = logFCcut, p.value = adjPcut)
res_voomLmFit_sr <- topTable(fit, sort.by = "P", n = Inf)

# Plot 2: voom
barplot(summary(dt_voom_sr)[c(1, 3), ], beside = TRUE, las = 2, 
        legend = TRUE, col = c("darkgreen", "darkred"),
        main = "voom")

library(EnhancedVolcano)
EnhancedVolcano(res_voomLmFit_sr,
    lab = rownames(res_voomLmFit_sr),
    x = 'logFC',
    y = 'P.Value',
    pCutoff = 0.01,
    FCcutoff = 1,
    title = "voom lm fit")

## Export results


In [ ]:

# merge results with counts
res_voomLmFit_sr$gene_id = rownames(res_voomLmFit_sr)
merged_sr = merge(res_voomLmFit_sr, as.data.frame(dge_filtered_sr$counts), by.x = "gene_id", by.y = "row.names", all.x = TRUE)

res_voomLmFit_lr_corr$gene_id = rownames(res_voomLmFit_lr_corr)
merged_lr_corr = merge(res_voomLmFit_lr_corr, as.data.frame(dge_filtered_lr_corr$counts), by.x = "gene_id", by.y = "row.names", all.x = TRUE)

In [ ]:
write.csv(merged_lr_corr, file = "LR_DESIREE_PAPER/ANALYSIS/final_tables_paper/voom_results_long_read_batch_corr_desiree.csv", row.names = TRUE)
write.csv(merged_sr, file = "LR_DESIREE_PAPER/ANALYSIS/final_tables_paper/voom_results_short_read_desiree.csv", row.names = TRUE)

## Compare log-fold changes

In [ ]:
library(ggpointdensity)
library(viridis)
library(gridExtra)
library(svglite)
library(dplyr)

options(repr.plot.width = 12, repr.plot.height = 6, repr.plot.res = 300)

# First set - corrected
intersect_genes_corr <- intersect(rownames(res_voomLmFit_lr_corr), rownames(res_voomLmFit_sr))
res_long_corr <- res_voomLmFit_lr_corr[intersect_genes_corr, ]
res_short_corr <- res_voomLmFit_sr[intersect_genes_corr, ]
logFC_long_corr <- res_long_corr$logFC
logFC_short_corr <- res_short_corr$logFC
correlation_corr <- cor(logFC_long_corr, logFC_short_corr)
my_data_corr <- data.frame(logFC_long = logFC_long_corr, logFC_short = logFC_short_corr)
print(paste("Correlation (corrected):", correlation_corr))

# Second set - uncorrected
intersect_genes_uncorr <- intersect(rownames(res_voomLmFit_lr_uncorr), rownames(res_voomLmFit_sr))
res_long_uncorr <- res_voomLmFit_lr_uncorr[intersect_genes_uncorr, ]
res_short <- res_voomLmFit_sr[intersect_genes_uncorr, ]
logFC_long_uncorr <- res_long_uncorr$logFC
logFC_short_uncorr <- res_short$logFC
correlation_uncorr <- cor(logFC_long_uncorr, logFC_short_uncorr)
my_data_uncorr <- data.frame(logFC_long = logFC_long_uncorr, logFC_short = logFC_short_uncorr)
print(paste("Correlation (uncorrected):", correlation_uncorr))

# Combine into one data frame with a panel variable (shared color scale)
my_data_corr$panel   <- "Batch Corrected"
my_data_uncorr$panel <- "Batch Uncorrected"
my_data_all <- rbind(my_data_uncorr, my_data_corr)
my_data_all$panel <- factor(my_data_all$panel,
                            levels = c("Batch Uncorrected", "Batch Corrected"))

# --- Per-panel annotation (Pearson r + n) ---
ann_all <- my_data_all %>%
  group_by(panel) %>%
  summarise(
    r = cor(logFC_short, logFC_long),
    n = n(),
    .groups = "drop"
  ) %>%
  mutate(label = paste0("Pearson r = ", sprintf("%.2f", r), "\nn = ", n))

FONTSIZE <- 20

# --- Draw ---
p_all <- ggplot(my_data_all, aes(x = logFC_short, y = logFC_long)) +
  geom_hex(bins = 30) +
  scale_fill_viridis(option = "turbo", trans = "log10", name = "count") +
  geom_label(data = ann_all,
             aes(x = -Inf, y = Inf, label = label),
             hjust = -0.05, vjust = 1.1,
             size = FONTSIZE - 13,
             fill = "white", alpha = 0.5, label.size = 0,
             inherit.aes = FALSE) +
  coord_fixed(ratio = 1, xlim = c(-5, 5), ylim = c(-5, 5)) +
  facet_wrap(~panel, nrow = 1) +
  labs(
    x = "short-read RNA-seq (log2FC(CPM))",
    y = "long-read RNA-seq (log2FC(CPM))"
  ) +
  theme_minimal(base_size = FONTSIZE) +
  theme(
    text = element_text(size = FONTSIZE),
    axis.text = element_text(size = FONTSIZE - 4),
    strip.text = element_text(size = FONTSIZE + 2, face = "bold"),
    legend.title = element_text(size = FONTSIZE - 2),
    legend.text = element_text(size = FONTSIZE - 4),
    panel.grid = element_blank(),
    axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black")
  )

options(repr.plot.width = 12, repr.plot.height = 6, repr.plot.res = 300)
p_all

# save as svg
# ggsave("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/LogFC_correlation_short_vs_long.svg",
#        plot = p_all, width = 12, height = 6, units = "in", dpi = 300)

In [ ]:
library(ggpointdensity)
library(viridis)
library(gridExtra)
library(svglite)
library(dplyr)

options(repr.plot.width = 7, repr.plot.height = 6, repr.plot.res = 300)

# Batch corrected
intersect_genes_corr <- intersect(rownames(res_voomLmFit_lr_corr), rownames(res_voomLmFit_sr))
res_long_corr <- res_voomLmFit_lr_corr[intersect_genes_corr, ]
res_short_corr <- res_voomLmFit_sr[intersect_genes_corr, ]
logFC_long_corr <- res_long_corr$logFC
logFC_short_corr <- res_short_corr$logFC
correlation_corr <- cor(logFC_long_corr, logFC_short_corr)
my_data_corr <- data.frame(logFC_long = logFC_long_corr, logFC_short = logFC_short_corr)
print(paste("Correlation (corrected):", correlation_corr))

# --- Annotation (Pearson r + n) ---
ann_corr <- data.frame(
  label = paste0("Pearson r = ", sprintf("%.2f", correlation_corr),
                 "\nn = ", nrow(my_data_corr))
)

FONTSIZE <- 20

# --- Draw ---
p_corr <- ggplot(my_data_corr, aes(x = logFC_short, y = logFC_long)) +
  geom_hex(bins = 30) +
  scale_fill_viridis(option = "turbo", trans = "log10", name = "count") +
  geom_label(data = ann_corr,
             aes(x = -Inf, y = Inf, label = label),
             hjust = -0.05, vjust = 1.1,
             size = FONTSIZE - 13,
             fill = "white", alpha = 0.5, label.size = 0,
             inherit.aes = FALSE) +
  coord_fixed(ratio = 1, xlim = c(-5, 5), ylim = c(-5, 5)) +
  labs(
    x = "short-read RNA-seq (log2FC(CPM))",
    y = "long-read RNA-seq (log2FC(CPM))"
  ) +
  theme_minimal(base_size = FONTSIZE) +
  theme(
    text = element_text(size = FONTSIZE),
    axis.text = element_text(size = FONTSIZE - 4),
    legend.title = element_text(size = FONTSIZE - 2),
    legend.text = element_text(size = FONTSIZE - 4),
    panel.grid = element_blank(),
    axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black")
  )

options(repr.plot.width = 7, repr.plot.height = 6, repr.plot.res = 300)
p_corr

# save as svg
ggsave("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/LogFC_Gene_correlation_short_vs_long_corrected.svg",
       plot = p_corr, width = 7, height = 6, units = "in", dpi = 300)

## plot correaltino for different gene lengths



In [ ]:
library(ggpointdensity)
library(viridis)
library(gridExtra)
library(svglite)
library(dplyr)

options(repr.plot.width = 18, repr.plot.height = 5, repr.plot.res = 300)

# --- Gene length table ---
gene_length_data <- read.table("LR_DESIREE_PAPER/DEG_results/avgTxlengths_desiree_unitato.tsv", header = TRUE)
gene_length_data$gene_id <- rownames(gene_length_data)
# rename columns
colnames(gene_length_data) <- c("AvgTranscriptLength", "gene_id")

# Batch corrected
intersect_genes_corr <- intersect(rownames(res_voomLmFit_lr_corr), rownames(res_voomLmFit_sr))
res_long_corr <- res_voomLmFit_lr_corr[intersect_genes_corr, ]
res_short_corr <- res_voomLmFit_sr[intersect_genes_corr, ]
logFC_long_corr <- res_long_corr$logFC
logFC_short_corr <- res_short_corr$logFC
correlation_corr <- cor(logFC_long_corr, logFC_short_corr)
my_data_corr <- data.frame(
  logFC_long  = logFC_long_corr,
  logFC_short = logFC_short_corr,
  gene_id     = rownames(res_long_corr)
)
print(paste("Correlation (corrected):", correlation_corr))

# --- Add gene length and bin it ---
my_data_corr <- merge(my_data_corr, gene_length_data, by = "gene_id")

my_data_corr <- my_data_corr %>%
  mutate(length_bin = cut(
    AvgTranscriptLength,
    breaks = c(0, 750, 1250, 3500, Inf),
    include.lowest = TRUE,
    labels = c("< 0.75 kb", "0.75-1.25 kb", "1.25-3.5 kb", "> 3.5 kb")
  ))

# --- Per-facet annotation (Pearson r + n) ---
ann_corr <- my_data_corr %>%
  group_by(length_bin) %>%
  summarise(
    r = cor(logFC_short, logFC_long),
    n = n(),
    .groups = "drop"
  ) %>%
  mutate(label = paste0("Pearson r = ", sprintf("%.2f", r), "\nn = ", n))

FONTSIZE <- 20

# --- Draw ---
p_corr <- ggplot(my_data_corr, aes(x = logFC_short, y = logFC_long)) +
  geom_hex(bins = 30) +
  scale_fill_viridis(option = "turbo", trans = "log10", name = "count") +
  geom_label(data = ann_corr,
             aes(x = -Inf, y = Inf, label = label),
             hjust = -0.05, vjust = 1.1,
             size = FONTSIZE - 13,
             fill = "white", alpha = 0.5, label.size = 0,
             inherit.aes = FALSE) +
  coord_fixed(ratio = 1, xlim = c(-5, 5), ylim = c(-5, 5)) +
  facet_wrap(~length_bin, nrow = 1) +
  labs(
    x = "short-read RNA-seq (log2FC(CPM))",
    y = "long-read RNA-seq (log2FC(CPM))"
  ) +
  theme_minimal(base_size = FONTSIZE) +
  theme(
    text = element_text(size = FONTSIZE),
    axis.text = element_text(size = FONTSIZE - 4),
    strip.text = element_text(size = FONTSIZE + 2, face = "bold"),
    legend.title = element_text(size = FONTSIZE - 2),
    legend.text = element_text(size = FONTSIZE - 4),
    panel.grid = element_blank(),
    axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black")
  )

options(repr.plot.width = 18, repr.plot.height = 5, repr.plot.res = 300)
p_corr

# save as svg
ggsave("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/LogFC_Gene_Length_correlation_short_vs_long_by_length.svg",
      plot = p_corr, width = 18, height = 5, units = "in", dpi = 300)

## Intersection plots

In [ ]:
# Define your gene sets
log2fc_threshold_long <- 1
padj_threshold_long <- 0.01
padj_threshold_short <- 0.01
log2fc_threshold_short <- 1

# Make sure variables are defined consistently with your earlier code
# or res_voomLmFit_lr_uncorr
# res_short is already defined from earlier

downregulated_genes_short <- rownames(res_short)[which(res_short$logFC < -log2fc_threshold_short & 
                                                       res_short$adj.P.Val < padj_threshold_short)]
upregulated_genes_short <- rownames(res_short)[which(res_short$logFC > log2fc_threshold_short & 
                                                     res_short$adj.P.Val < padj_threshold_short)]
downregulated_genes_corr <- rownames(res_long_corr)[which(res_long_corr$logFC < -log2fc_threshold_long & 
                                           res_long_corr$adj.P.Val < padj_threshold_long)]
upregulated_genes_corr <- rownames(res_long_corr)[which(res_long_corr$logFC > log2fc_threshold_long & 
                                         res_long_corr$adj.P.Val < padj_threshold_long)]
downregulated_genes_uncorr <- rownames(res_long_uncorr)[which(res_long_uncorr$logFC < -log2fc_threshold_long & 
                                           res_long_uncorr$adj.P.Val < padj_threshold_long)]
upregulated_genes_uncorr <- rownames(res_long_uncorr)[which(res_long_uncorr$logFC > log2fc_threshold_long & 
                                         res_long_uncorr$adj.P.Val < padj_threshold_long)]                                        

# Get all unique genes from both result sets
all_genes <- unique(c(rownames(res_long_corr), rownames(res_long_uncorr), rownames(res_short)))

# Create comprehensive results table
results_table <- data.frame(
  gene_id = all_genes,
  Illumina_DEG_down = all_genes %in% downregulated_genes_short,
  Illumina_DEG_up = all_genes %in% upregulated_genes_short,
  Kinnex_DEG_down = all_genes %in% downregulated_genes_corr,
  Kinnex_DEG_up = all_genes %in% upregulated_genes_corr,
  Kinnex_DEG_down_uncorr = all_genes %in% downregulated_genes_uncorr,
  Kinnex_DEG_up_uncorr = all_genes %in% upregulated_genes_uncorr,
  stringsAsFactors = FALSE
)

# Create comprehensive results table
results_table_combined <- data.frame(
  gene_id = all_genes,
  Illumina_DEG = all_genes %in% downregulated_genes_short | all_genes %in% upregulated_genes_short,
  Kinnex_DEG = all_genes %in% downregulated_genes_corr | all_genes %in% upregulated_genes_corr,
  Kinnex_DEG_uncorr = all_genes %in% downregulated_genes_uncorr | all_genes %in% upregulated_genes_uncorr,
  stringsAsFactors = FALSE
)

# Store raw results before adding additional columns
results_table_raw <- results_table

In [ ]:
length(upregulated_genes_uncorr)
library(UpSetR)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 300)
# Select expression categories
expression_categories <- colnames(results_table)[c(2, 3, 4, 5)]

# Prepare data for UpSet plot
results_table_corr <- results_table[expression_categories]
results_table_corr <- results_table_corr[rowSums(results_table_corr) > 0, ]

# Convert to logical/binary
results_table_corr[expression_categories] <- results_table_corr[expression_categories] == 1

# Increase plot size and margins
par(cex = 1.5, cex.axis = 1.3, cex.lab = 1.3)

# Create UpSet plot
upset(results_table_corr, 
      expression_categories, 
      width_ratio = 0.2)

In [ ]:

# export the results to a CSV file
write.csv(results_table[,1:5], file = "LR_DESIREE_PAPER/ANALYSIS/final_tables_paper/DEG_overlap_short_vs_long_up_down_corr.tsv", row.names = FALSE, quote = FALSE)


In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 300)
# Select expression categories
expression_categories <- colnames(results_table_combined)[c(2, 3)]

# Prepare data for UpSet plot
results_table_corr <- results_table_combined[expression_categories]
results_table_corr <- results_table_corr[rowSums(results_table_corr) > 0, ]

# Convert to logical/binary
results_table_corr[expression_categories] <- results_table_corr[expression_categories] == 1

# Increase plot size and margins
par(cex = 1.5, cex.axis = 1.3, cex.lab = 1.3)

# Create UpSet plot
upset(results_table_corr, 
      expression_categories, 
      width_ratio = 0.2)

### venn diagramm

In [ ]:
library(VennDiagram)
library(grid)

# --- Compute areas ---
area1      <- sum(results_table_corr[[expression_categories[1]]])
area2      <- sum(results_table_corr[[expression_categories[2]]])
cross.area <- sum(results_table_corr[[expression_categories[1]]] &
                  results_table_corr[[expression_categories[2]]])

# --- Open SVG device ---
svg(
  "LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DEG_overlap_short_vs_long_venn.svg",
  width  = 3.5,
  height = 3.5
)

# --- Draw ---
grid.newpage()
draw.pairwise.venn(
  area1      = area1,
  area2      = area2,
  cross.area = cross.area,
  category   = expression_categories,
  fill       = c("darkorange2", "grey"),
  alpha      = 0.5,

  # Numbers inside circles
  cex        = 1.5,
  fontface   = "bold",
  fontfamily = "sans",

  # Category labels
  cat.cex        = 1.3,
  cat.fontface   = "bold",
  cat.fontfamily = "sans",
  cat.pos        = c(-20, 20),   # nudge labels outward symmetrically
  cat.dist       = c(0.05, 0.05),

  # Avoid label overlap with ext.text
  ext.text  = TRUE,
  ext.line.lwd = 1.5,
  ext.dist     = -0.15,
  ext.length   = 0.85,

  # Circle outlines
  lwd   = 2,
  lty   = "solid",
  col   = c("black", "black")   # darker border than fill
)

# --- Close device ---
dev.off()

### Mercator enrichment

In [ ]:

# get the gene names for DEGs in Kinnex and Illumina
Illumina_DEGs_only <- results_table_combined$gene_id[results_table_combined$Illumina_DEG ]
Kinnex_DEGs_only <- results_table_combined$gene_id[results_table_combined$Kinnex_DEG ]
Kinnex_Illumina_DEGs <- results_table_combined$gene_id[results_table_combined$Kinnex_DEG & results_table_combined$Illumina_DEG]
geneUnivserse = results_table_combined$gene_id
# write the table to a file
write.table(results_table_combined, file = "LR_DESIREE_PAPER/ANALYSIS/final_tables_paper/DEG_overlap_short_vs_long.tsv", sep = "\t", row.names = FALSE, quote = FALSE)

In [ ]:
library(clusterProfiler)
mercator = read.gmt("reference/ADAPT_liftoff/UniTato_Mercator4_gene-level.gmt")

In [ ]:
ans.tf_Kinnex_only <- enricher(Kinnex_DEGs_only, TERM2GENE=mercator, universe=geneUnivserse, minGSSize=3, maxGSSize=500, qvalueCutoff=0.01, pvalueCutoff = 0.01)
p1 <- barplot(ans.tf_Kinnex_only, font.size=5, title="Kinnex DEGs Mercator Enrichment", width = 0.5)
p1
# make a list of enriched terms for Kinnex only DEGs
enriched_terms_Kinnex_only <- subset(ans.tf_Kinnex_only@result, p.adjust < 0.01, qvalue > 0.02)$Description
#also add p.adjust and qvalue to the list
enriched_terms_Kinnex_only <- subset(ans.tf_Kinnex_only@result)[, c("Description", "p.adjust")]
length(enriched_terms_Kinnex_only)

In [ ]:
length(geneUnivserse)

In [ ]:
ans.tf_Illumina_only <- enricher(Illumina_DEGs_only, TERM2GENE=mercator, universe = geneUnivserse, minGSSize=3, maxGSSize=500, qvalueCutoff=0.01, pvalueCutoff = 0.01)
p2 <- barplot(ans.tf_Illumina_only, font.size=5, title="Illumina phased DEGs",width = 0.5)

# grid arrange the two plots side by side
grid.arrange(p1, p2, nrow = 1)
enriched_terms_Illumina_only <- subset(ans.tf_Illumina_only@result)[, c("Description", "p.adjust")]

length(enriched_terms_Illumina_only)


In [ ]:
# export the list of enriched terms to a text file
write.table(enriched_terms_Kinnex_only, file = "LR_DESIREE_PAPER/ANALYSIS/final_tables_paper/enriched_terms_DM_Kinnex_only.txt", sep = "\t", row.names = FALSE, quote = FALSE)
write.table(enriched_terms_Illumina_only, file = "LR_DESIREE_PAPER/ANALYSIS/final_tables_paper/enriched_terms_DM_Illumina_only.txt", sep = "\t", row.names = FALSE, quote = FALSE)

In [ ]:
subset(ans.tf@result, p.adjust < 0.05)

## Heat map of heat shock proteins

In [ ]:
heat_shock_proteins <-  c(
  "Soltu.DM.08G012580", 
  "Soltu.DM.08G012670",
  "Soltu.DM.08G024980",
  "Soltu.DM.08G025000",
  "Soltu.DM.09G020330",
  "Soltu.DM.09G020340",
  "Soltu.DM.09G020350",
  "Soltu.DM.09G025880",
"Soltu.DM.09G025890",
"Soltu.DM.09G025900",
"Soltu.DM.09G025910",
"Soltu.DM.09G0259202",
"Soltu.DM.09G025930",
"Soltu.DM.09G025940"
  
)

In [ ]:


# Convert the VST counts to long format for ggplot2
library(reshape2)

# First compare wide vs long version
sample_names = names(corrected_data)[3:length(names(corrected_data))]
# cpm and log2 transform    
corrected_data_cpm <- corrected_data
corrected_data_cpm[, sample_names] <- cpm(corrected_data[, sample_names], log=TRUE, prior.count=0.5)
corrected_data_long <- melt(corrected_data_cpm, id.vars=c("gene_id", "gene_name"), measure.vars=sample_names, variable.name="variable", value.name="value")

# subset to heat shock proteins
corrected_data_long <- corrected_data_long[corrected_data_long$gene_id %in% heat_shock_proteins, ]
# convert value to cpm 

# remove counts that are less than 1 in all samples


# Make a heatmap
heatmap <- ggplot(corrected_data_long, aes(x=variable, y=gene_id, fill=value)) + geom_raster() + theme(axis.text.x=element_text(angle=65, hjust=1), axis.text.y=element_blank(), axis.ticks.y=element_blank())
heatmap

In [ ]:
library(reshape2)

# First compare wide vs long version
sample_names = names(corrected_data)[3:length(names(corrected_data))]
# cpm and log2 transform    
corrected_data_cpm <- uncorrected_data
corrected_data_cpm[, sample_names] <- cpm(uncorrected_data[, sample_names], log=TRUE, prior.count=1)
corrected_data_long <- melt(corrected_data_cpm, id.vars=c("gene_id", "gene_name"), measure.vars=sample_names, variable.name="variable", value.name="value")

# subset to heat shock proteins
corrected_data_long <- corrected_data_long[corrected_data_long$gene_id %in% heat_shock_proteins, ]
# convert value to cpm 

# Make a heatmap
heatmap <- ggplot(corrected_data_long, aes(x=variable, y=gene_id, fill=value)) + geom_raster()  + theme(axis.text.x=element_text(angle=65, hjust=1), axis.text.y=element_blank(), axis.ticks.y=element_blank())
heatmap

In [ ]:
# First compare wide vs long version
sample_names = names(corrected_data)[3:length(names(corrected_data))]
# cpm and log2 transform    
corrected_data_cpm <- as.data.frame(dge_sr$counts)
corrected_data_cpm$gene_id <- rownames(dge_sr$counts)
corrected_data_cpm[, sample_names] <- cpm(uncorrected_data[, sample_names], log=TRUE, prior.count=1)
corrected_data_long <- melt(corrected_data_cpm, id.vars=c("gene_id"), measure.vars=sample_names, variable.name="variable", value.name="value")

# subset to heat shock proteins
corrected_data_long <- corrected_data_long[corrected_data_long$gene_id %in% heat_shock_proteins, ]
# convert value to cpm 

# Make a heatmap
heatmap <- ggplot(corrected_data_long, aes(x=variable, y=gene_id, fill=value)) + geom_raster()  + theme(axis.text.x=element_text(angle=65, hjust=1), axis.text.y=element_blank(), axis.ticks.y=element_blank())
heatmap

## multimapping genes



In [ ]:
# read in the unique and ambig counts

ambig_counts <- read.table("Desiree_ASE/output/nf_core_rnaseq_short_read_heat_unitato/salmon/GENE_COUNTS/DEHH19_heat.counts.tsv", header = TRUE)
ambig_counts_2 <- read.table("Desiree_ASE/output/nf_core_rnaseq_short_read_heat_unitato/salmon/GENE_COUNTS/DECH13_control.counts.tsv", header = TRUE)
# summ the ambig and unique counts
ambig_counts['UniqueCount'] = ambig_counts['UniqueCount'] + ambig_counts_2['UniqueCount']
ambig_counts['AmbigCount'] = ambig_counts['AmbigCount'] + ambig_counts_2['AmbigCount']
#rename colnames 
colnames(ambig_counts) <- c("transcript_id", "UniqueCount", "AmbigCount")

tx2gene <- read.table("Desiree_ASE/output/nf_core_rnaseq_short_read_heat_unitato/salmon/tx2gene.tsv", header = TRUE)

# merge counts and transcripts
merged_counts <- merge(ambig_counts, tx2gene, by = "transcript_id")
# now group by gene_id and sum unique counts per gene, for ambig counts take the average
final_counts <- merged_counts %>%
  group_by(gene_id) %>%
  summarise(UniqueCount = sum(UniqueCount), AmbigCount = mean(AmbigCount))

# remove genes with less then 5 unique and ambig counts
final_counts <- final_counts %>%
  filter(UniqueCount + AmbigCount >= 5)
# add the multimapping ratio per gene
final_counts <- final_counts %>%
  mutate(UniqueMappingRatio = UniqueCount / (AmbigCount + UniqueCount))

# add a multimapping category
final_counts <- final_counts %>%
  mutate(MultiMappingCategory = case_when(
    UniqueMappingRatio < 0.25 ~ "Low Multimapping",
    UniqueMappingRatio < 0.8 ~ "High Multimapping",
    TRUE ~ NA
  ))

low_multimapping_genes_list = final_counts$gene_id[final_counts$MultiMappingCategory == "Low Multimapping"]


In [ ]:
library(ggpointdensity)
library(viridis)
library(gridExtra)
library(svglite)
# save as svg with svg lite
# svglite(
#   "LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/LogFC_phased_correlation_short_vs_long.svg",
#   width  = 10,
#   height = 4
# )
options(repr.plot.width = 10, repr.plot.height = 4, repr.plot.res = 300)
# First plot - corrected
intersect_genes_corr <- intersect(rownames(res_voomLmFit_lr_corr), rownames(res_voomLmFit_sr))
res_long_corr <- res_voomLmFit_lr_corr[intersect_genes_corr, ]
res_short_corr <- res_voomLmFit_sr[intersect_genes_corr, ]
# add the multimapping category to the results data frame
res_long_corr$unique_mapping_ratio <- final_counts$UniqueMappingRatio[match(rownames(res_long_corr), final_counts$gene_id)]

logFC_long_corr <- res_long_corr$logFC
logFC_short_corr <- res_short_corr$logFC  
correlation_corr <- cor(logFC_long_corr, logFC_short_corr)
my_data_corr <- data.frame(logFC_long = logFC_long_corr, logFC_short = logFC_short_corr, unique_mapping_ratio = res_long_corr$unique_mapping_ratio)
print(paste("Correlation (corrected):", correlation_corr))


library(dplyr)

my_data_corr <- my_data_corr %>%
  mutate(mapping_bin = cut(
    unique_mapping_ratio,
    breaks = c(0, 0.1, 0.5, 0.75, 1),
    include.lowest = TRUE,
    labels = c("0–10%", "10–50%", "50–75%", "75–100%")
  ))
# --- Draw ---

# add the number of points in the plot
n_points_corr <- nrow(my_data_corr)
p1 <- ggplot(my_data_corr,
             aes(x = logFC_short,
                 y = logFC_long)) +
  geom_point(alpha = 0.2, size = 1.5) +
  geom_pointdensity() +
  scale_color_viridis()  +  
  coord_cartesian(xlim = c(-5, 5), ylim = c(-5, 5)) +
  facet_wrap(~mapping_bin, ncol = 2) +
  labs(
    x = "Log2FC Illumina",
    y = "Log2FC Kinnex"
  ) +
  theme_minimal() +
  theme(
    text = element_text(size = 16),
    strip.text = element_text(size = 14, face = "bold")
  )

# Combine plots side-by-side
#grid.arrange(p1, ncol = 1)
options(repr.plot.width = 5, repr.plot.height = 5, repr.plot.res = 300)
p1
# save as svg
#ggsave("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/LogFC_correlation_short_vs_long.svg", width = 20, height = 8, units = "in", dpi = 300)

In [ ]:
options(repr.plot.width = 3, repr.plot.height = 3, repr.plot.res = 300)
# make a histogram of multimapping ratio per regulation category
ggplot(final_counts, aes(x = UniqueMappingRatio)) +
  geom_histogram(binwidth = 0.1, position = "dodge") +
  labs(title = "Unique Mapping Ratio",
       x = "Unique Mapping Ratio",
       y = "Count") +
  theme_minimal()

# check for gene length differences

In [ ]:

gene_length_data <- read.table("LR_DESIREE_PAPER/DEG_results/avgTxlengths_desiree_unitato.tsv", header = TRUE)
gene_length_data$gene_id = rownames(gene_length_data)
# rename columns
colnames(gene_length_data) <- c("AvgTranscriptLength", "gene_id")
merge(final_results_table, gene_length_data, by = "gene_id") -> final_results_table_with_length

In [ ]:
# order categories 
# not significant, deg in short only, deg in long only, deg in both
final_results_table_with_length$RegulationCategory <- factor(final_results_table_with_length$RegulationCategory, levels = c("Not significant", "DEG in both", "DEG in Illumina only", "DEG in Kinnex only"))

# plot only "DEG in short only", "DEG in long only"
final_results_table_subset <- final_results_table_with_length %>%
  filter(RegulationCategory %in% c("DEG in Illumina only", "DEG in Kinnex only", "DEG in both"))
options(repr.plot.width = 5, repr.plot.height = 2, repr.plot.res = 300)


clean_subset <- remove_missing(final_results_table_subset, vars = "RegulationCategory")

ggplot(data = clean_subset) +
  geom_density(aes(x = UniqueMappingRatio, color = RegulationCategory), alpha = 1, adjust = 1/2) +
  scale_color_brewer(palette = "Dark2") +
  geom_vline(
    data = clean_subset %>%
      group_by(RegulationCategory) %>%
      summarise(mean_ratio = mean(UniqueMappingRatio, na.rm = TRUE)),
    aes(xintercept = mean_ratio, color = RegulationCategory),
    linetype = "dashed"
  ) + theme_minimal()


In [ ]:
# get the mean unique mapping ratio per category
final_results_table_subset %>%
  group_by(RegulationCategory) %>%
  summarise(mean_unique_mapping_ratio = mean(UniqueMappingRatio, na.rm = TRUE))

In [ ]:
library(ggplot2)
library(ggridges)

options(repr.plot.width = 5, repr.plot.height = 3, repr.plot.res = 300)

ggplot(final_results_table_with_length, aes(x = AvgTranscriptLength, 
                                              y = RegulationCategory, 
                                              fill = RegulationCategory)) +
  geom_density_ridges(alpha = 0.7, 
                      scale = 1.2, 
                      rel_min_height = 0.01,
                      bandwidth = 200,
                      jittered_points = FALSE,
                      panel_scaling = TRUE) +
  labs(title = "Distribution of Average Transcript Length by Regulation Category",
       x = "Average Transcript Length (bp)",
       y = "Regulation Category") +
  theme_ridges(grid = TRUE) +
  theme(text = element_text(size = 14),
        plot.title = element_text(size = 14, face = "bold"),
        axis.title = element_text(size = 14),
        axis.text = element_text(size = 12),
        axis.text.y = element_text(vjust = 0),
        legend.position = "none") +
  xlim(0, 5000) +
  scale_fill_brewer(palette = "Set2") +
  scale_y_discrete(expand = c(0, 0))

In [ ]:
# order categories 
# not significant, deg in short only, deg in long only, deg in both
final_results_table_with_length$RegulationCategory <- factor(final_results_table_with_length$RegulationCategory, levels = c("Not significant", "DEG in both", "DEG in Illumina only", "DEG in Kinnex only"))
# plot only "DEG in short only", "DEG in long only"
final_results_table_with_length_subset <- final_results_table_with_length %>%
  filter(RegulationCategory %in% c("DEG in Illumina only", "DEG in Kinnex only", "DEG in both"))
options(repr.plot.width = 5, repr.plot.height = 2, repr.plot.res = 300)


# also add mean lines
ggplot(data = remove_missing((final_results_table_with_length_subset), vars = "RegulationCategory")) + 
  geom_density(aes(x = AvgTranscriptLength, color = RegulationCategory), alpha = 1) + 
  xlim(0, 6000) +
  scale_color_brewer(palette = "Dark2") +
  geom_vline(
    data = final_results_table_with_length_subset %>%
      group_by(RegulationCategory) %>%
      summarise(mean_length = mean(AvgTranscriptLength, na.rm = TRUE)),
    aes(xintercept = mean_length, color = RegulationCategory),
    linetype = "dashed"
  ) + theme_minimal()
  # add white style 


In [ ]:
# add a violin plot of gene length per category
# remove genes that have avg transcipt length greater than 20000
final_results_table_with_length <- final_results_table_with_length %>%
  filter(AvgTranscriptLength < 20000)
options(repr.plot.width = 10, repr.plot.height = 6, repr.plot.res = 100)
ggplot(final_results_table_with_length, aes(x = RegulationCategory, y = AvgTranscriptLength, fill = RegulationCategory)) +
  geom_violin(alpha = 0.7) +
  geom_jitter(width = 0.2, size = 0.05, alpha = 0.3) +
  stat_summary(fun = mean, geom = "point", shape = 18, size = 3, color = "black") +
  labs(title = "Average Transcript Length by Regulation Category",
       x = "Regulation Category",
       y = "Average Transcript Length (bp)") +
  theme_minimal() +
  theme(text = element_text(size = 14),
        plot.title = element_text(size = 16, face = "bold"),
        axis.title = element_text(size = 16),
        axis.text = element_text(size = 14),
        legend.position = "none") #+ ylim(0, 20000) +

  scale_fill_brewer(palette = "Set2")   

In [ ]:
# group by regulation category and calculate mean gene length per category
mean_gene_length <- final_results_table_with_length %>%
  group_by(RegulationCategory) %>%
  summarise(mean_length = mean(AvgTranscriptLength, na.rm = TRUE))
#final_results_table_with_length